In [ ]:
# ── Environment Setup ─────────────────────────────────────────────────────
import os
import sys
from pathlib import Path

# Optional: force using GPU 0 (useful on multi-GPU machines)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Set this to False if you intentionally want to run locally.
KAGGLE_ONLY = True
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ONLY and not IS_KAGGLE:
    raise RuntimeError(
        "This notebook is configured to run on Kaggle only.\n"
        "Open it on Kaggle, or set KAGGLE_ONLY=False in the first cell."
    )

WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
os.chdir(WORK_DIR)
RESULTS_DIR = WORK_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# ── Install libraries (Kaggle) ─────────────────────────────────────────────
if IS_KAGGLE:
    os.system(
        "pip install transformers datasets peft trl bitsandbytes accelerate sacrebleu sentencepiece -q"
    )

# ── Hugging Face Token ────────────────────────────────────────────────────
HF_TOKEN = None

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if HF_TOKEN is None:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face login: OK")
else:
    print("WARNING: HF_TOKEN is not set.")
    print("  Kaggle: Add-ons → Secrets → add HF_TOKEN")
    print("  Or set env var HF_TOKEN")

# ── Detect accelerator ───────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("\nEnvironment : Kaggle" if IS_KAGGLE else "\nEnvironment : Local")
print(f"DEVICE      : {DEVICE}")
print(f"WORK_DIR    : {WORK_DIR}")
print(f"RESULTS_DIR : {RESULTS_DIR}")

# Experiment 2 — Fine-tune an LLM (Gemma-2-2B) with QLoRA

**Goal:** Fine-tune a decoder-only LLM for **English → Vietnamese** translation using a **LoRA adapter** (QLoRA: 4-bit base model + LoRA).

**Runs on:** Google Colab (T4) | Kaggle (T4/P100) | Local (CUDA/MPS/CPU)

**Outputs:** LoRA adapter folder, training logs, BLEU score, and prediction samples in `results/`.

In [ ]:
# Check GPU
if DEVICE == "cuda":
    !nvidia-smi
else:
    print(f"No NVIDIA GPU — running on: {DEVICE}")

## Block 1: Imports

Libraries used in this notebook:

- `peft`: LoRA adapters
- `trl`: `SFTTrainer` for supervised fine-tuning
- `bitsandbytes`: 4-bit quantization (QLoRA)
- `accelerate`: device mapping / multi-GPU utilities

In [ ]:
import json
import torch
import sacrebleu
from datasets import load_from_disk, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print("PyTorch version:", torch.__version__)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Block 2: Model + Quantization

**QLoRA workflow:**

1. Load the base model in **4-bit** (significantly lower VRAM).
2. Prepare the model for k-bit training.
3. Add a **LoRA adapter** (train a small fraction of parameters).

In [ ]:
model_name = "google/gemma-2-2b-it"

# 4-bit quantization config
# Tesla T4 typically requires float16 instead of bfloat16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},        # Force the entire model onto GPU 0
    token=HF_TOKEN,
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {model_name}")
print(f"Model dtype: {model.dtype}")

## Block 3: Load Data

In [ ]:
from pathlib import Path
import shutil
import os

# Automatically find the data directory
if IS_KAGGLE:
    print("Searching for data in Kaggle Input...")
    # Find all directories named 'iwslt15_cleaned' inside /kaggle/input
    found_paths = list(Path("/kaggle/input").rglob("iwslt15_cleaned"))
    
    if not found_paths:
        raise FileNotFoundError(
            "Could not find the 'iwslt15_cleaned' directory! "
            "Make sure you have clicked 'Add Data' and added the output of Notebook 01."
        )
    
    source_path = str(found_paths[0])
    dataset_path = "/kaggle/working/iwslt15_cleaned"
    
    # Copy to a writable directory (/kaggle/working)
    if not os.path.exists(dataset_path):
        print(f"Copying data from {source_path} to {dataset_path} to avoid Read-Only error...")
        shutil.copytree(source_path, dataset_path)
    else:
        print(f"Data already copied to {dataset_path}")

else:
    dataset_path = "data/processed/iwslt15_cleaned"

# Load data
dataset = load_from_disk(dataset_path)

print(dataset)
print("\n--- Inspect 1 sample ---")
print("Fields:", dataset["train"].column_names)
sample = dataset["train"][0]
print("EN:", sample["en"])
print("VI:", sample["vi"])

## Block 4: LoRA Configuration

LoRA trains ~1–2% of parameters instead of the full model:

- `r=16`: rank of the low-rank decomposition (higher = more capacity)
- `lora_alpha=16`: scaling factor (often set equal to `r`)
- `target_modules`: attention + MLP projection layers to inject LoRA adapters

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: trainable params ~1–2% of total params

## Block 5: Prompt Formatting

Why formatting differs from Experiment 1:

- Encoder–Decoder models consume separate input/output sequences.
- Decoder-only LLMs learn from **a single text sequence**: instruction → expected answer.
- Here we use the Gemma chat format: `<start_of_turn>user/model ...`

In [ ]:
def format_translation_prompt(example):
    """Convert an EN/VI pair into a Gemma-style instruction prompt."""
    en = example["en"]
    vi = example["vi"]

    text = (
        f"<start_of_turn>user\n"
        f"Translate the following English text to Vietnamese:\n"
        f"{en}<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"{vi}<end_of_turn>"
    )
    return {"text": text}

# Format the full dataset
train_dataset = dataset["train"].map(format_translation_prompt, desc="Formatting train")
val_dataset = dataset["validation"].map(format_translation_prompt, desc="Formatting val")

# Sanity-check the format
print("Sample prompt:")
print(train_dataset[0]["text"])

## Block 6: Training Configuration

Notes:

- `fp16=False`, `bf16=False` → train in float32 (slower, but avoids bf16 issues on T4).
- `max_steps=3000` → cap training steps instead of running a full epoch.
- `save_steps=500` → checkpoint frequently to survive disconnects.

In [ ]:
training_args = SFTConfig(
    output_dir="./results-llm-lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    max_steps=3000,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    warmup_steps=100,
    report_to="none",
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    dataset_text_field="text",
    max_length=256,
    ddp_find_unused_parameters=False,
    dataloader_pin_memory=False,
)

print("Training config OK.")
print(f"  max_steps={training_args.max_steps}, save_steps={training_args.save_steps}")

## Block 7: Train with `SFTTrainer`

`SFTTrainer` (Supervised Fine-Tuning Trainer) from TRL is designed for instruction fine-tuning of decoder-only LLMs.

Estimated time: **~3 hours** on a T4 (float32).

If you get disconnected, you can resume with:
`trainer.train(resume_from_checkpoint="./results-llm-lora/checkpoint-xxx")`

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

# If disconnected, uncomment and replace xxx with the latest checkpoint number:
# trainer.train(resume_from_checkpoint="./results-llm-lora/checkpoint-xxx")
trainer.train()

In [ ]:
# Save LoRA adapter (only a few MBs, not the whole model)
model.save_pretrained("./gemma-2b-en-vi-lora")
tokenizer.save_pretrained("./gemma-2b-en-vi-lora")
print("Saved LoRA adapter to ./gemma-2b-en-vi-lora")

# Save training log
train_log = trainer.state.log_history
with open(str(RESULTS_DIR / "training_log_exp2.json"), "w") as f:
    json.dump(train_log, f, indent=2)
print("Saved training_log_exp2.json")

# Free memory
torch.cuda.empty_cache()

## Block 8: Inference + Evaluation

Translate the test set and compute BLEU. We use greedy decoding (`do_sample=False`) for consistent, reproducible outputs.

In [ ]:
# Prepare test data
source_texts = dataset["test"]["en"]
references = dataset["test"]["vi"]

print(f"Test samples: {len(source_texts)}")
print(f"Example source: {source_texts[0]}")
print(f"Example reference: {references[0]}")

In [ ]:
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

llm_predictions = []

for idx, src in enumerate(source_texts):
    prompt = (
        f"<start_of_turn>user\n"
        f"Translate the following English text to Vietnamese:\n"
        f"{src}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,      # Greedy decoding for consistency
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Get only the generated output (exclude prompt input)
    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    llm_predictions.append(generated.strip())

    if idx % 100 == 0:
        print(f"  Processed {idx}/{len(source_texts)}")

# Calculate BLEU
llm_bleu = sacrebleu.corpus_bleu(llm_predictions, [references])
print(f"\nLLM Fine-tuned BLEU: {llm_bleu.score:.2f}")

In [ ]:
# Save intermediate predictions checkpoint
with open(str(RESULTS_DIR / "llm_predictions_checkpoint.json"), "w", encoding="utf-8") as f:
    json.dump({
        "bleu": llm_bleu.score,
        "predictions": llm_predictions[:50]
    }, f, ensure_ascii=False, indent=2)
print("Saved llm_predictions_checkpoint.json")

## Block 9: Save Results

Save metrics to `results/exp2_llm_lora.json` and save 20 qualitative examples to `results/llm_translation_examples.json`.

In [ ]:
# Save evaluation metrics
results_data = {
    "model": "google/gemma-2-2b-it + QLoRA",
    "bleu": llm_bleu.score,
    "hyperparameters": {
        "lora_r": 16,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "learning_rate": 2e-4,
        "max_steps": 3000,
        "batch_size": "4 x 4 (gradient accumulation)",
        "quantization": "4-bit NF4",
        "max_seq_length": 256,
        "warmup_steps": 100,
        "optimizer": "paged_adamw_32bit",
    },
    "dataset": "iwslt15_cleaned",
    "test_samples": len(source_texts),
}

with open(str(RESULTS_DIR / "exp2_llm_lora.json"), "w") as f:
    json.dump(results_data, f, indent=2)
print("Saved exp2_llm_lora.json")
print(json.dumps(results_data, indent=2))

In [ ]:
# Save 20 translation examples for qualitative analysis (Day 3)
examples = []
for i in range(20):
    examples.append({
        "source": source_texts[i],
        "reference": references[i],
        "llm_translation": llm_predictions[i],
    })

with open(str(RESULTS_DIR / "llm_translation_examples.json"), "w", encoding="utf-8") as f:
    json.dump(examples, f, ensure_ascii=False, indent=2)
print("Saved llm_translation_examples.json")

# Print a few examples for verification
for ex in examples[:3]:
    print("\nSRC:", ex["source"])
    print("REF:", ex["reference"])
    print("LLM:", ex["llm_translation"])

In [ ]:
# Summary
print("=" * 50)
print("EXPERIMENT 2 COMPLETED")
print("=" * 50)
print(f"LLM Fine-tuned BLEU: {llm_bleu.score:.2f}")
print(f"\nFiles saved:")
print(f"  {RESULTS_DIR}/exp2_llm_lora.json")
print(f"  {RESULTS_DIR}/llm_translation_examples.json")
print(f"  {RESULTS_DIR}/llm_predictions_checkpoint.json")
print(f"  {RESULTS_DIR}/training_log_exp2.json")
print(f"  ./gemma-2b-en-vi-lora/  (LoRA adapter)")